# 🚀 Day 9 — LLM Prompting, Embeddings, RAG & MLOps Finale
**Week 2 Finale | Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | LLM Prompting Patterns & Embeddings | ~10 sec |
| 2 | 10:30–11:00 AM | Semantic Search & RAG Concept | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Prompt Engineering + Full MLOps | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Week 2 Finale: Full MLOps Pipeline | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, re, json, hashlib, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve
)
from sklearn.datasets import load_breast_cancer, load_wine, load_iris
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — LLM Prompting Patterns & Word Embeddings
**Time:** 9:00–10:30 AM | **Run time: ~10 sec**

In [ ]:
# 1.1  Token Counting Intuition (without an LLM)
# Approximate BPE tokenisation using whitespace + punctuation split
def approx_tokenise(text):
    tokens = re.findall(r"[\w']+|[.,!?;:]", text.lower())
    return tokens

sentences = [
    "Machine learning is transforming artificial intelligence research.",
    "Transformers use self-attention to process sequences in parallel.",
    "ChatGPT is a large language model trained on internet-scale data.",
]
for s in sentences:
    toks = approx_tokenise(s)
    print(f'Words: {len(s.split()):3d}  Approx tokens: {len(toks):3d}  Text: "{s[:50]}..."')

print('\nKey insight: tokens < words (punctuation separate) or > words (subword splits)')
print('Rule of thumb: 1 word ≈ 1.3 tokens in English')

In [ ]:
# 1.2  Word Embeddings via Co-occurrence + SVD
corpus = [
    "machine learning models train on data",
    "deep learning is a subset of machine learning",
    "neural networks learn features from data automatically",
    "data science uses statistics and programming skills",
    "python is the most popular language for data analysis",
    "gradient descent optimises machine learning models",
    "random forests and neural networks are powerful models",
    "clustering groups similar data points together",
    "supervised learning requires labelled training data",
    "unsupervised learning finds patterns without labels",
]

# Build co-occurrence matrix
cv = CountVectorizer(min_df=1, max_features=50)
X_doc_term = cv.fit_transform(corpus)
cooc_matrix = (X_doc_term.T @ X_doc_term).toarray().astype(float)
np.fill_diagonal(cooc_matrix, 0)  # remove self-co-occurrence

# SVD to get dense embeddings
svd = TruncatedSVD(n_components=8, random_state=42)
embeddings = svd.fit_transform(cooc_matrix)
vocab = list(cv.get_feature_names_out())

print(f'Vocabulary size: {len(vocab)}')
print(f'Embedding shape: {embeddings.shape}  (words × dimensions)')
print(f'Variance explained by 8 dims: {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# 1.3  Semantic Similarity in Embedding Space
def most_similar(word, top_k=4):
    if word not in vocab:
        return []
    idx  = vocab.index(word)
    sims = cosine_similarity([embeddings[idx]], embeddings)[0]
    sims[idx] = -1  # exclude self
    top  = np.argsort(sims)[::-1][:top_k]
    return [(vocab[i], round(sims[i], 4)) for i in top]

query_words = ['learning', 'data', 'models', 'python']
print('Semantic neighbours in embedding space:')
for word in query_words:
    neighbours = most_similar(word)
    print(f'  {word:10s} → {neighbours}')

In [ ]:
# 1.4  Visualise Embedding Space with PCA
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 7))
colors_by_domain = {
    'learning':'#534AB7','machine':'#534AB7','deep':'#534AB7','neural':'#534AB7',
    'data':'#1D9E75','science':'#1D9E75','analysis':'#1D9E75','statistics':'#1D9E75',
    'models':'#D85A30','networks':'#D85A30','forests':'#D85A30','clustering':'#D85A30',
}
for i, word in enumerate(vocab):
    color = colors_by_domain.get(word, '#888780')
    ax.scatter(coords[i, 0], coords[i, 1], color=color, s=60, alpha=0.8)
    ax.annotate(word, (coords[i, 0], coords[i, 1]),
                xytext=(4, 4), textcoords='offset points', fontsize=9)
from matplotlib.patches import Patch
legend = [
    Patch(color='#534AB7', label='Learning concepts'),
    Patch(color='#1D9E75', label='Data concepts'),
    Patch(color='#D85A30', label='Model concepts'),
    Patch(color='#888780', label='Other'),
]
ax.legend(handles=legend, fontsize=9)
ax.set_title('Word Embedding Space (PCA 2D) — semantic clusters visible')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

---
## Session 2 — Semantic Search & RAG
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  TF-IDF Retriever (RAG backbone)
knowledge_base = [
    "Random forests build multiple decision trees and average their predictions.",
    "Gradient boosting trains trees sequentially, each correcting the previous.",
    "Neural networks use backpropagation and gradient descent to learn weights.",
    "SVMs find the hyperplane that maximises the margin between class boundaries.",
    "K-means assigns each point to the nearest centroid and updates centroids iteratively.",
    "PCA reduces dimensionality by projecting data onto principal components.",
    "Cross-validation estimates model performance on unseen data reliably.",
    "Regularisation prevents overfitting by penalising large model weights.",
    "Feature scaling ensures all inputs contribute equally to distance metrics.",
    "Precision measures how often positive predictions are correct.",
    "Recall measures how many actual positives the model correctly identified.",
    "AUC-ROC measures classifier ranking ability across all decision thresholds.",
]

tfidf_retriever = TfidfVectorizer(ngram_range=(1,2))
doc_vecs = tfidf_retriever.fit_transform(knowledge_base)

def retrieve(query, k=3, verbose=True):
    q_vec = tfidf_retriever.transform([query])
    sims  = cosine_similarity(q_vec, doc_vecs)[0]
    top_k = np.argsort(sims)[::-1][:k]
    results = [(knowledge_base[i], round(float(sims[i]), 4)) for i in top_k]
    if verbose:
        print(f'Query: "{query}"')
        for doc, score in results:
            print(f'  [{score:.4f}] {doc}')
        print()
    return results

queries = [
    "how do ensemble methods combine models?",
    "what metric should I use for imbalanced classes?",
    "how does regularisation stop overfitting?",
]
for q in queries:
    retrieve(q, k=2)

In [ ]:
# 2.2  Simple RAG Loop (retrieve → inject context → simulate answer)
def rag_answer(query, k=2):
    """Simulate RAG: retrieve docs, build context prompt, print result."""
    retrieved = retrieve(query, k=k, verbose=False)
    context   = '\n'.join([f'- {doc}' for doc, _ in retrieved])

    # Build the prompt that would go to an LLM
    prompt = f"""You are an ML tutor. Use only the context below to answer.

Context:
{context}

Question: {query}

Answer:"""
    print(f'=== RAG Prompt for: "{query}" ===')
    print(prompt)
    print()
    return prompt

rag_answer("how do ensemble methods combine models?")
rag_answer("what metric for imbalanced classification?")

In [ ]:
# 2.3  Chunking Strategies Comparison
long_doc = """
Machine learning is a subset of artificial intelligence. It enables computers to learn 
from data without being explicitly programmed. Supervised learning uses labelled examples. 
Unsupervised learning finds patterns without labels. Reinforcement learning trains agents 
through rewards and penalties. Deep learning uses neural networks with many layers. 
Transformers revolutionised NLP with attention mechanisms. BERT uses bidirectional encoding. 
GPT uses autoregressive generation. Fine-tuning adapts pretrained models to new tasks.
"""

# Strategy 1: Sentence-level chunks
sentence_chunks = [s.strip() for s in long_doc.split('.') if len(s.strip()) > 10]

# Strategy 2: Fixed-size word chunks (window=15, stride=10)
words = long_doc.split()
fixed_chunks = [' '.join(words[i:i+15]) for i in range(0, len(words)-15, 10)]

print(f'Sentence chunks: {len(sentence_chunks)}')
for i, c in enumerate(sentence_chunks[:3]):
    print(f'  [{i+1}] {c[:70]}')

print(f'\nFixed-size chunks (w=15, stride=10): {len(fixed_chunks)}')
for i, c in enumerate(fixed_chunks[:3]):
    print(f'  [{i+1}] {c[:70]}')

---
## Session 3 — Prompt Engineering + MLOps Dashboard
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Prompt Engineering Patterns
class PromptBuilder:
    """Structured prompt template builder."""

    @staticmethod
    def zero_shot(task, query):
        return f"Task: {task}\nInput: {query}\nOutput:"

    @staticmethod
    def few_shot(task, examples, query):
        ex_str = '\n'.join([f'  Input: {i}\n  Output: {o}' for i, o in examples])
        return f"Task: {task}\n\nExamples:\n{ex_str}\n\nInput: {query}\nOutput:"

    @staticmethod
    def chain_of_thought(task, query):
        return (
            f"Task: {task}\n"
            f"Input: {query}\n"
            "Let's think step by step:\n"
            "Step 1:"
        )

    @staticmethod
    def system_user(system, user):
        return f"[SYSTEM]\n{system}\n\n[USER]\n{user}\n\n[ASSISTANT]"

pb = PromptBuilder()

print('=== Zero-Shot ===')
print(pb.zero_shot('Classify sentiment as POS/NEG', 'The movie was absolutely brilliant!'))

print('\n=== Few-Shot ===')
print(pb.few_shot(
    'Classify sentiment as POS/NEG',
    [('Great product!', 'POS'), ('Terrible service.', 'NEG'), ('It was okay.', 'NEG')],
    'Really enjoyed this book'
))

print('\n=== Chain-of-Thought ===')
print(pb.chain_of_thought(
    'Decide if a loan applicant is low/high risk',
    'Age: 35, Income: 60000, Debt: 5000, Credit score: 720'
))

print('\n=== System+User ===')
print(pb.system_user(
    system='You are a data scientist. Be concise and technical.',
    user='Explain overfitting in one sentence.'
))

In [ ]:
# 3.2  Self-Consistency Voting (simulate multiple LLM samples)
import random
random.seed(42)

def simulate_cot_samples(question, n_samples=7):
    """
    Simulate self-consistency: sample n reasoning paths,
    take majority vote on the final answer.
    """
    # Mock: each sample independently reaches an answer with some noise
    true_answer = 'POSITIVE'
    answers = []
    for i in range(n_samples):
        # 80% chance of correct answer per sample
        answer = true_answer if random.random() < 0.80 else 'NEGATIVE'
        steps  = f'Step {i+1}: analysed sentiment cues... → {answer}'
        answers.append((steps, answer))
    return answers

q = 'Is "This ML course changed my career!" positive or negative?'
samples = simulate_cot_samples(q)
final_answers = [a for _, a in samples]
from collections import Counter
vote = Counter(final_answers).most_common(1)[0]

print(f'Question: {q}')
print(f'\n{len(samples)} sampled reasoning paths:')
for step, ans in samples:
    print(f'  {step}')
print(f'\nMajority vote: {vote[0]} ({vote[1]}/{len(samples)} paths agree)')

In [ ]:
# 3.3  MLOps Metrics Dashboard
rng = np.random.default_rng(42)
N_DAYS = 60

# Simulate production metrics over 60 days
days = np.arange(1, N_DAYS+1)
drift = days * 0.0012
auc_prod    = (0.945 - drift + rng.normal(0, 0.004, N_DAYS)).clip(0.7, 1.0)
latency_p50 = 18 + rng.exponential(2, N_DAYS)
latency_p99 = latency_p50 * 3.5 + rng.exponential(5, N_DAYS)
throughput  = 1200 + rng.integers(-80, 80, N_DAYS)
error_rate  = (0.001 + drift * 0.002 + rng.exponential(0.0005, N_DAYS)).clip(0, 0.05)

# Simulate retraining events
retrain_days = [20, 42]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# AUC over time
axes[0,0].plot(days, auc_prod, color='#534AB7', linewidth=1.5)
for rd in retrain_days:
    axes[0,0].axvline(rd, color='#1D9E75', linestyle='--', linewidth=1.5, alpha=0.8)
axes[0,0].axhline(0.91, color='#D85A30', linestyle=':', label='Alert threshold')
axes[0,0].set_title('Model AUC — Production'); axes[0,0].set_xlabel('Day')
axes[0,0].legend(fontsize=8); axes[0,0].grid(alpha=0.3)

# Latency percentiles
axes[0,1].fill_between(days, latency_p50, latency_p99, alpha=0.2, color='#534AB7')
axes[0,1].plot(days, latency_p50, color='#534AB7', linewidth=1.5, label='p50')
axes[0,1].plot(days, latency_p99, color='#D85A30', linewidth=1.5, label='p99')
axes[0,1].axhline(100, color='red', linestyle=':', label='p99 SLA=100ms')
axes[0,1].set_title('Serving Latency (ms)'); axes[0,1].set_xlabel('Day')
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# Throughput
axes[1,0].bar(days, throughput, color='#1D9E75', alpha=0.7, width=0.8)
axes[1,0].axhline(throughput.mean(), color='black', linestyle='--', linewidth=1,
                   label=f'Mean={throughput.mean():.0f} req/min')
axes[1,0].set_title('Request Throughput'); axes[1,0].set_xlabel('Day')
axes[1,0].legend(fontsize=8)

# Error rate
axes[1,1].plot(days, error_rate * 100, color='#D85A30', linewidth=1.5)
axes[1,1].axhline(1.0, color='red', linestyle=':', label='1% alert')
axes[1,1].set_title('Error Rate (%)'); axes[1,1].set_xlabel('Day')
axes[1,1].legend(fontsize=8); axes[1,1].grid(alpha=0.3)

plt.suptitle('MLOps Production Dashboard — 60 Days (green lines = retraining)', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Average AUC     : {auc_prod.mean():.4f}')
print(f'p99 breaches    : {(latency_p99 > 100).sum()} days')
print(f'Error rate peak : {error_rate.max():.2%}')

---
## Lunch Break — 1:00–2:00 PM
1. Why does co-occurrence + SVD capture semantic meaning?
2. RAG vs fine-tuning — when should you choose each?
3. What does temperature=0 do to an LLM output?
4. Name the 5 stages of the MLOps lifecycle.
---

## Session 5 — Week 2 Finale: Full MLOps Pipeline
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Feature Store Pattern
class FeatureStore:
    """Cache precomputed features with versioning and metadata."""
    def __init__(self):
        self._store   = {}
        self._meta    = {}

    def write(self, entity_id, features, version='v1', tags=None):
        key = f'{entity_id}:{version}'
        self._store[key] = np.array(features)
        self._meta[key]  = {
            'timestamp': datetime.utcnow().isoformat(),
            'shape'    : np.array(features).shape,
            'tags'     : tags or [],
        }

    def read(self, entity_id, version='v1'):
        key = f'{entity_id}:{version}'
        return self._store.get(key), self._meta.get(key)

    def batch_read(self, entity_ids, version='v1'):
        arrays = []
        for eid in entity_ids:
            feat, _ = self.read(eid, version)
            if feat is not None:
                arrays.append(feat)
        return np.vstack(arrays) if arrays else None

    def summary(self):
        return pd.DataFrame([
            {'key': k, **v, 'shape': str(v['shape'])}
            for k, v in self._meta.items()
        ])

# Demo
fs = FeatureStore()
rng = np.random.default_rng(0)
for i in range(10):
    fs.write(f'user_{i}', rng.normal(0,1,15).round(4), version='v1', tags=['numerical'])

print('Feature store created with 10 user feature vectors')
feat, meta = fs.read('user_3')
print(f'user_3 features shape: {feat.shape}')
print(f'user_3 metadata: {meta}')
print(f'\nFeature store summary:')
print(fs.summary().head())

In [ ]:
# 5.2  Batch Scoring + Online Serving Simulation
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_tr)
X_tr_sc = scaler.transform(X_tr)
X_te_sc  = scaler.transform(X_te)

model = RandomForestClassifier(100, random_state=42).fit(X_tr_sc, y_tr)

# --- Batch scoring ---
t0 = time.perf_counter()
batch_probs = model.predict_proba(X_te_sc)[:, 1]
batch_time  = (time.perf_counter() - t0) * 1000
print(f'Batch scoring: {len(X_te)} samples in {batch_time:.2f}ms  '
      f'({batch_time/len(X_te):.3f}ms/sample)')

# --- Online serving simulation (single-sample latency) ---
latencies = []
for i in range(500):
    sample = X_te_sc[[i % len(X_te_sc)]]
    t0 = time.perf_counter()
    model.predict_proba(sample)
    latencies.append((time.perf_counter() - t0) * 1000)

lat = np.array(latencies)
print(f'\nOnline serving latency (500 requests):')
for pct in [50, 90, 95, 99]:
    print(f'  p{pct:>2}: {np.percentile(lat, pct):.3f}ms')
print(f'  max: {lat.max():.3f}ms')

SLA_P99 = 5.0  # ms
print(f'\nSLA p99 < {SLA_P99}ms: {"PASS" if np.percentile(lat,99) < SLA_P99 else "FAIL"} '
      f'(actual p99={np.percentile(lat,99):.3f}ms)')

In [ ]:
# 5.3  Complete MLOps Feedback Loop
class MLOpsPipeline:
    """
    End-to-end MLOps: data → features → train → eval → serve → monitor → retrain
    """
    def __init__(self, name):
        self.name      = name
        self.versions  = []
        self.champion  = None

    def ingest_and_featurise(self, X, y):
        """Step 1: Normalise and create feature hash."""
        scaler = StandardScaler().fit(X)
        X_sc   = scaler.transform(X)
        dhash  = hashlib.md5(X.tobytes()).hexdigest()[:8]
        return X_sc, y, scaler, dhash

    def train(self, X, y, dhash):
        """Step 2: Train model on processed features."""
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42)
        model   = RandomForestClassifier(80, class_weight='balanced', random_state=42)
        model.fit(X_tr, y_tr)
        metrics = {
            'auc'      : round(roc_auc_score(y_va, model.predict_proba(X_va)[:,1]), 4),
            'accuracy' : round(model.score(X_va, y_va), 4),
            'n_train'  : len(X_tr),
        }
        version = {'v': len(self.versions)+1, 'metrics': metrics, 'data_hash': dhash,
                   'ts': datetime.utcnow().isoformat()[:19], 'model': model}
        self.versions.append(version)
        return version

    def evaluate_and_promote(self, version):
        """Step 3: Promote if AUC beats current champion."""
        if self.champion is None or version['metrics']['auc'] > self.champion['metrics']['auc']:
            self.champion = version
            return True, 'Promoted to champion'
        return False, f'Champion retained (AUC {self.champion["metrics"]["auc"]} >= {version["metrics"]["auc"]})'

    def serve(self, X_new, scaler):
        """Step 4: Predict using champion model."""
        return self.champion['model'].predict_proba(scaler.transform(X_new))[:, 1]

    def monitor_drift(self, train_scores, prod_scores, threshold=0.15):
        """Step 5: Check PSI drift."""
        breaks = np.unique(np.percentile(train_scores, np.linspace(0,100,11)))
        e = np.histogram(train_scores, bins=breaks)[0] / len(train_scores) + 1e-8
        a = np.histogram(prod_scores,  bins=breaks)[0] / len(prod_scores)  + 1e-8
        psi = float(np.sum((a - e) * np.log(a / e)))
        return psi, psi > threshold

    def report(self):
        print(f'=== MLOps Pipeline: {self.name} ===')
        for v in self.versions:
            marker = ' [CHAMPION]' if v is self.champion else ''
            print(f'  v{v["v"]} | AUC={v["metrics"]["auc"]} | '
                  f'Acc={v["metrics"]["accuracy"]} | data={v["data_hash"]} | {v["ts"]}{marker}')

# Run the full pipeline
pipeline = MLOpsPipeline('breast-cancer-classifier')
rng = np.random.default_rng(1)

# Round 1: initial training
X1, y1, sc1, h1 = pipeline.ingest_and_featurise(X_bc, y_bc)
v1 = pipeline.train(X1, y1, h1)
promoted, reason = pipeline.evaluate_and_promote(v1)
print(f'Round 1: {reason}')

# Round 2: new data arrives, potential drift
X2_raw = np.vstack([X_bc, rng.normal(X_bc.mean(0), X_bc.std(0)*1.1, (50, X_bc.shape[1]))])
y2     = np.concatenate([y_bc, rng.integers(0,2,50)])
X2, y2, sc2, h2 = pipeline.ingest_and_featurise(X2_raw, y2)
v2 = pipeline.train(X2, y2, h2)
promoted, reason = pipeline.evaluate_and_promote(v2)
print(f'Round 2: {reason}')

# Monitor drift
train_s = pipeline.champion['model'].predict_proba(X1)[:,1]
prod_s  = train_s + rng.normal(0.1, 0.05, len(train_s))
psi, trigger = pipeline.monitor_drift(train_s, prod_s)
print(f'Drift PSI: {psi:.4f} | Retrain triggered: {trigger}')

pipeline.report()

In [ ]:
# 5.4  Week 2 Grand Finale: All Systems Summary
print('='*55)
print(' WEEK 2 — FINAL LEADERBOARD')
print('='*55)

datasets = {
    'Breast Cancer' : load_breast_cancer(return_X_y=True),
    'Wine'          : load_wine(return_X_y=True),
    'Iris'          : load_iris(return_X_y=True),
}

models = {
    'LogisticReg'  : make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
    'RandomForest' : RandomForestClassifier(80, random_state=42),
    'GradBoost'    : GradientBoostingClassifier(60, random_state=42),
}

results_final = []
for ds_name, (Xd, yd) in datasets.items():
    for m_name, m in models.items():
        s = cross_val_score(m, Xd, yd, cv=5, scoring='accuracy')
        results_final.append({'Dataset':ds_name,'Model':m_name,'CV Acc':round(s.mean(),4),'Std':round(s.std(),4)})

df_final = pd.DataFrame(results_final)
pivot    = df_final.pivot(index='Model', columns='Dataset', values='CV Acc')
print(pivot.round(4))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(pivot.columns))
width = 0.25
colors = ['#534AB7','#1D9E75','#D85A30']
for i, (model_name, color) in enumerate(zip(pivot.index, colors)):
    ax.bar(x + i*width, pivot.loc[model_name], width, label=model_name, color=color, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(pivot.columns)
ax.set_ylabel('CV Accuracy'); ax.set_ylim(0.85, 1.01)
ax.set_title('Week 2 Grand Finale — Model × Dataset Leaderboard')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('\nDay 9 & Week 2 complete!')
print('You have covered Days 1-9: NumPy → sklearn → ensembles → deep learning')
print('→ deployment → graphs → LLMs → full MLOps. You are production-ready!')

---
## Day 9 Summary

| Concept | Skill Gained |
|---|---|
| Token intuition | Word vs token count, BPE approximation |
| Word embeddings | Co-occurrence matrix, SVD, semantic similarity |
| Embedding visualisation | PCA 2D projection, semantic cluster inspection |
| TF-IDF retriever | Cosine similarity ranking, top-k retrieval |
| RAG pipeline | Retrieve context → inject into prompt → answer |
| Chunking strategies | Sentence vs fixed-size window with stride |
| Prompt patterns | Zero-shot, few-shot, chain-of-thought, system+user |
| Self-consistency | Multiple CoT samples, majority vote aggregation |
| MLOps dashboard | AUC, latency percentiles, throughput, error rate |
| Feature store | Write/read/batch with versioning and metadata |
| Batch + online serving | Throughput vs latency, p50/p95/p99, SLA check |
| Full MLOps loop | Ingest → featurise → train → eval → serve → monitor → retrain |

---
## Week 2 Complete!
**9 days covered:**
- Days 1–5: NumPy, Pandas, sklearn, ensembles, deployment, capstone
- Days 6–9: Math/stats, recommenders, fraud, graph ML, LLMs, full MLOps

You are now equipped to contribute to real production ML systems.

In [ ]:
# Bonus: Week 2 in 10 lines
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(100, random_state=42).fit(Xtr, ytr)
auc   = roc_auc_score(yte, model.predict_proba(Xte)[:,1])
print(f'Week 2 final model AUC: {auc:.4f}')

# RAG in 3 lines
docs = ['Random forests use bagging', 'Boosting trains sequentially', 'SVMs maximise margin']
tv = TfidfVectorizer().fit(docs)
best = docs[cosine_similarity(tv.transform(['ensemble trees']), tv.transform(docs))[0].argmax()]
print(f'RAG retrieved: "{best}"')
print('Week 2 complete — 9 days, production-ready ML!')